In [ ]:
import os
import sys
import shutil
import ee

print("Initializing environment setup...")

# 1. CLEANUP PREVIOUS INSTALLATIONS
# Remove potential residual files to ensure a clean import
if os.path.exists('/content/tagee.py'):
    os.remove('/content/tagee.py')

if os.path.exists('/content/tagee_repo'):
    shutil.rmtree('/content/tagee_repo')

# 2. INSTALL LIBRARY
# Install the 'tagee' package directly from the source repository
print("Installing 'tagee' library...")
!pip install git+https://github.com/zecojls/tagee.git --force-reinstall -q

# 3. VERIFY IMPORT
try:
    import tagee
    from tagee import terrainAnalysis
    print(f"Library loaded successfully from: {tagee.__file__}")
except ImportError as e:
    print("CRITICAL ERROR: Failed to import 'tagee' module.")
    print("SOLUTION: Restart the Runtime session and execute this cell again.")
    raise e

# 4. GEE AUTHENTICATION
# USER CONFIGURATION: Replace the string below with your Google Cloud Project ID.
# This is required for API access.
USER_PROJECT_ID = 'YOUR_GOOGLE_CLOUD_PROJECT_ID_HERE'

print(f"Authenticating with Project ID: {USER_PROJECT_ID}...")

if USER_PROJECT_ID == 'YOUR_GOOGLE_CLOUD_PROJECT_ID_HERE':
    print("WARNING: You have not set a valid Project ID. The script may fail.")

try:
    ee.Initialize(project=USER_PROJECT_ID)
    print("Google Earth Engine initialized successfully.")
except:
    print("Authentication required. Please follow the instructions in the prompt...")
    ee.Authenticate()
    ee.Initialize(project=USER_PROJECT_ID)
    print("Authentication complete.")

In [ ]:
import math
import ee
from tagee import terrainAnalysis

# --- USER CONFIGURATION SECTION ---

# 1. INPUT DATA (FeatureCollection)
# Replace this path with the Asset ID of your vector data (e.g., 'users/your_name/your_watershed')
INPUT_ASSET_ID = 'users/your_username/your_shapefile_asset'

# 2. ANALYSIS SCALE (Resolution)
# Set the resolution in meters.
# - Use 30 for native NASADEM resolution (faster).
# - Use lower values (e.g., 10 or 5) for oversampling or if using high-res local DEMs.
ANALYSIS_SCALE = 10

# ----------------------------------

# --- DEM SETUP ---
# Using NASADEM for global coverage and improved vertical accuracy over SRTM
dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')

# Apply Gaussian smoothing to reduce DEM noise artifacts
# Parameters: radius=3, sigma=2 (Safanelli et al., 2020)
gaussianFilter = ee.Kernel.gaussian(radius=3, sigma=2, units='pixels', normalize=True)
srtmSmooth = dem.convolve(gaussianFilter)

# --- LOAD USER DATA ---
print(f"Loading FeatureCollection from: {INPUT_ASSET_ID}")
user_collection = ee.FeatureCollection(INPUT_ASSET_ID)

# --- TERRAIN METRICS CALCULATION ---
# Compute morphometric variables using the 'tagee' algorithm
terrainMetrics = terrainAnalysis(srtmSmooth)

# Define Reducer: Calculates both Mean and Standard Deviation
reducer = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.stdDev(), sharedInputs=True
)

# Rename bands to abbreviated codes for cleaner attribute tables
abbreviated_metrics = terrainMetrics.select(
    ['Elevation', 'Slope', 'Aspect', 'Hillshade', 'Northness', 'Eastness',
     'HorizontalCurvature', 'VerticalCurvature', 'MeanCurvature', 'MinimalCurvature',
     'MaximalCurvature', 'GaussianCurvature', 'ShapeIndex'],
    ['El', 'S', 'As', 'Hill', 'Nor', 'Eas', 'HCv', 'VCv',
     'MeCur', 'MinCur', 'MaxCur', 'Gauss', 'Sh']
)

# --- ZONAL STATISTICS FUNCTION ---
def calculate_stats(feature):
    """
    Computes statistical summaries for a single feature geometry.
    Uses reduceRegion with the user-defined scale.
    """
    stats = abbreviated_metrics.reduceRegion(
        reducer=reducer,
        geometry=feature.geometry(),
        scale=ANALYSIS_SCALE,
        maxPixels=1e13, # High limit to accommodate complex geometries
        tileScale=4,    # Optimization for memory management
        bestEffort=False
    )
    return feature.set(stats)

print("Analysis logic configured. Ready for export.")

In [ ]:
# --- EXPORT CONFIGURATION (SINGLE FILE) ---

OUTPUT_FOLDER = 'GEE_Terrain_Analysis'  # Folder name in Google Drive
OUTPUT_FILENAME = 'Terrain_Metrics_Complete'

print("Initializing Single File Export...")

# Apply the zonal statistics function to the entire collection
processed_collection = user_collection.map(calculate_stats)

# Define the Export Task
task = ee.batch.Export.table.toDrive(
    collection=processed_collection,
    description=OUTPUT_FILENAME,
    folder=OUTPUT_FOLDER,
    fileFormat='GeoJSON'
)

task.start()
print(f"Export Task '{OUTPUT_FILENAME}' submitted to Google Earth Engine.")
print(f"Destination Drive Folder: {OUTPUT_FOLDER}")

In [ ]:
# --- EXPORT CONFIGURATION (BATCH) ---

BATCH_SIZE = 1000   # Number of features per file
OUTPUT_FOLDER = 'GEE_Terrain_Analysis_Batch'
BASE_FILENAME = 'Terrain_Metrics_Part'

# 1. Compute Batch Requirements
try:
    total_count = user_collection.size().getInfo()
    num_batches = math.ceil(total_count / BATCH_SIZE)

    print(f"Starting Batch Process: {total_count} features will be split into {num_batches} files.")

    # 2. Execution Loop
    for i in range(num_batches):
        start_index = i * BATCH_SIZE

        # Slicing the collection
        batch_list = user_collection.toList(BATCH_SIZE, start_index)
        batch_col = ee.FeatureCollection(batch_list)

        # Processing the slice
        batch_processed = batch_col.map(calculate_stats)

        # Formatting filename (e.g., Part_01)
        file_desc = f"{BASE_FILENAME}_{i+1:02d}"

        # Submitting Task
        task = ee.batch.Export.table.toDrive(
            collection=batch_processed,
            description=file_desc,
            folder=OUTPUT_FOLDER,
            fileFormat='GeoJSON'
        )

        task.start()
        print(f"Submitted Task: {file_desc}")

    print("All batch tasks have been submitted successfully.")

except Exception as e:
    print(f"Error initializing batch export. Details: {e}")

In [ ]:
# --- TASK CANCELLATION UTILITY ---
import ee

print("Checking for active tasks...")

tasks = ee.batch.Task.list()
cancelled_count = 0

for task in tasks:
    # Check for RUNNING or READY states
    if task.status()['state'] in ['RUNNING', 'READY']:
        # Filter based on the filename prefix used in previous cells
        if 'Terrain_Metrics' in task.config['description']:
            task.cancel()
            print(f"[CANCELED] ID: {task.id} | Desc: {task.config['description']}")
            cancelled_count += 1

if cancelled_count == 0:
    print("No matching active tasks found.")
else:
    print(f"Cancelled {cancelled_count} tasks.")

In [ ]:
# --- FORMAT CONVERSION: GEOJSON TO SHAPEFILE ---
import os
import glob
from google.colab import drive

# 1. Install Dependencies
try:
    import geopandas as gpd
except ImportError:
    print("Installing Geopandas...")
    !pip install geopandas -q
    import geopandas as gpd

# 2. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 3. Path Configuration
# UPDATE THIS if you used a different folder name in Cell 3 or 4
TARGET_DRIVE_FOLDER = 'GEE_Terrain_Analysis_Batch'

base_path = '/content/drive/MyDrive'
input_dir = os.path.join(base_path, TARGET_DRIVE_FOLDER)
output_dir = os.path.join(input_dir, 'Shapefiles')

# Create output directory
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 4. Conversion Loop
geojson_files = glob.glob(os.path.join(input_dir, "*.geojson"))
print(f"Found {len(geojson_files)} GeoJSON files.")

for f_path in geojson_files:
    try:
        filename = os.path.basename(f_path)
        name_clean = os.path.splitext(filename)[0]
        shp_path = os.path.join(output_dir, name_clean + ".shp")

        if os.path.exists(shp_path):
            print(f"[SKIP] Exists: {name_clean}.shp")
            continue

        print(f"[CONVERTING] {filename}...")
        gdf = gpd.read_file(f_path)
        gdf.to_file(shp_path)

    except Exception as e:
        print(f"[ERROR] Could not convert {filename}: {e}")

print("Conversion process finished.")